# Argument realization in an Ojibwe corpus: parsing the treebank

In this notebook, we will take an `.xml` formatted corpus file and turn it into a `.conllu` formatted treebank.

### Part 1: Parsing the xml file
We will begin by taking a parsed Ojibwe text in `.xml` format, and storing Ojibwe and English sentence pairs in a list variable `sentences`. 

**N.B.** The Ojibwe and English sentences in each paragraph are assumed to stand in a one-to-one relationship, which is admittedly a coarse heuristic, meaning there may be mismatches. The English sentences are primarily for reference when working with the treebank data.

In [ ]:
from grammar_modules.disambiguation import REPO_ROOT
import xml.etree.ElementTree as ET
import re

XML_PATH = REPO_ROOT / "data" / "corpus" / "Chi_mewinzha" / "2026-08-16_Chi-mewinzha.xml"

tree = ET.parse(XML_PATH) 
root = tree.getroot()

total_sent_count = 0
sentences = [] # tuple of matching (oj, en) sentence pairs
en_sents = 0
for section in root.findall(".//section"):
    current_sec = section.get("num")
    for ch_i, chapter in enumerate(section.findall("chapter")):
        current_ch = chapter.get("num")
        for paragraph in chapter.findall("paragraph"):
            current_para = paragraph.get("num")
            eng_para = paragraph.find("text_eng").text
            if eng_para is not None:
                eng_sentences = re.split(r'(?<=[.!?])\s*', eng_para)
                eng_sentences = [s for s in eng_sentences if s]
            else:
                eng_sentences = []

            
            oj_sentences = paragraph.findall("text_ojb/sentence")
            for sent_i, sent in enumerate(oj_sentences):
                total_sent_count+=1
                # find sent_text node
                sent_text_node = sent.find("sent_text")
                # get actual sentence text
                oj_sent_text = sent_text_node.text.strip() if sent_text_node.text is not None else None
                if oj_sent_text is None: continue

                current_sent = sent.get("num")
                if len(eng_sentences) > sent_i: 
                    en_sent_text = eng_sentences[sent_i]
                else: 
                    en_sent_text = "No English translation."

                sentences.append((oj_sent_text, en_sent_text))
                print(f"Sec ", current_sec, "Ch ", current_ch, "Para ", current_para, "Sent ", current_sent, "Overall sent ", total_sent_count, "Paragraph", "Oj sentence:", oj_sent_text, "En translation:", en_sent_text)
                print("-" * 40)




Sec  1 Ch  1 Para  1 Sent  1 Overall sent  1 Paragraph Oj sentence: Aanish niwii-tibaajim i'iw Gaa-zagaskwaajimekaag. En translation:  Well, I want to talk about Leech Lake.
----------------------------------------
Sec  1 Ch  1 Para  1 Sent  2 Overall sent  2 Paragraph Oj sentence: Mii widi wenjibaayaan idi Gaa-zagaskwaajimekaag. En translation: That’s where I’m from, Leech Lake.
----------------------------------------
Sec  1 Ch  1 Para  1 Sent  3 Overall sent  3 Paragraph Oj sentence: Naaniibowa gaawiin ogikendanziinaawaa iw baataniinowag imaa ozagaskwaajimeg, ge-sh giiwenh gaa-izhiwebak jibwaa-dagoshinowaad chi-mookomaanag imaa. En translation: Lots of people don’t know that there are a lot of leeches in there, and also what happened before the white man came.
----------------------------------------
Sec  1 Ch  1 Para  1 Sent  4 Overall sent  4 Paragraph Oj sentence: Nimaamaanaan iko niwiindamaagonaanig gaa-inakamigak iidog. En translation: Our mom would tell us about what happened.

### Part 2: Running the treebank pipeline

We now run each sentence through the dependency parsing pipeline to create a `.conllu` formatted treebank. The logic is in `treebank_modules/corpus.py`.The xml contains disambiguated FST readings, but we will re-run sentences through the pipeline, so that we keep the parsing local to this repository.

In [3]:
from grammar_modules.disambiguation import REPO_ROOT
from treebank_modules.corpus import build_treebank_from_xml_sentences

corpus_path = REPO_ROOT / "data" / "treebanks" / "2026_08_17_Chi_mewinzha.conllu"
num_sentences = build_treebank_from_xml_sentences(sentences=sentences, corpus_path=corpus_path, batch_size=500)

FST file is /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/OjibweMorph-v1_2_1.fomabin
Processed batch 1: 500 sentences added.
Processed batch 2: 634 sentences added.
Added 634 sentences to 2026_08_17_Chi_mewinzha.conllu.
